In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from umap import UMAP

from climate_attitudes.datasets.reduced_no_imputation import schema
from climate_attitudes.visualisation import configure_mpl

RANDOM_SEED = 202606041925

rng = np.random.default_rng(RANDOM_SEED)


np.set_printoptions(linewidth=200)

configure_mpl(Path("../fonts"))

DATA_PATH = Path("../reports/thesis/results/data/model/all_interventions/")

schema = schema.post_index()

In [ ]:
labels = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")["labels"]

inits = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")["measurements"][:, :, 0][
    ..., 0
]
inits_orig = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")["Y0"][:, 1]

no_int_asym_outcome = np.load(DATA_PATH / "ising_00_no_use_covariates.npz")[
    "measurements"
][:, :, 5]
int_asym_outcome = np.load(DATA_PATH / "ising_15_no_use_covariates.npz")[
    "measurements"
][:, :, 5]

int_effect = int_asym_outcome - no_int_asym_outcome

individual_int_effect_on_policy = int_effect[:, :, 7].mean(axis=1)

Looking at CC worry (idx 2) and climate impacts (idx 6)

In [ ]:
individual_effect_ccw_ccp = individual_int_effect_on_policy[:, 2]
individual_effect_cci_ccp = individual_int_effect_on_policy[:, 6]

In [ ]:
def get_percentile_idxes(a, plim):
    vlow, vhigh = np.percentile(a, plim)
    print(vlow, vhigh)
    return np.argwhere((a >= vlow) & (a <= vhigh)).flatten()

In [ ]:
ccw_ccp_lo_eff_idxes = get_percentile_idxes(individual_effect_ccw_ccp, (0, 10))
ccw_ccp_hi_eff_idxes = get_percentile_idxes(individual_effect_ccw_ccp, (90, 100))

cci_ccp_lo_eff_idxes = get_percentile_idxes(individual_effect_cci_ccp, (0, 10))
cci_ccp_hi_eff_idxes = get_percentile_idxes(individual_effect_cci_ccp, (90, 100))

Check overlap --- 32%.

In [ ]:
(
    len(set(ccw_ccp_lo_eff_idxes) & set(cci_ccp_lo_eff_idxes))
    / len(set(ccw_ccp_lo_eff_idxes) | set(cci_ccp_lo_eff_idxes))
)

For a given set of individuals, calculate the probability that each initial spin state is 1

In [ ]:
def init_spin_probs(idxes, inits):
    print((inits[idxes] == np.ones(inits.shape[-1])).mean(axis=(0, 1)))

In [ ]:
init_spin_probs(ccw_ccp_lo_eff_idxes, inits)

In [ ]:
init_spin_probs(ccw_ccp_hi_eff_idxes, inits)

In [ ]:
labels

## Who is 'CC Worry' intervention ineffective for?

In [ ]:
umap = UMAP()

X = np.concat((inits_orig[ccw_ccp_lo_eff_idxes], (inits_orig[ccw_ccp_hi_eff_idxes])))
Y = np.concat((np.zeros(ccw_ccp_lo_eff_idxes.size), np.ones(ccw_ccp_hi_eff_idxes.size)))

embs_ccw_ccp = umap.fit_transform(X, y=Y)

# embs_lo_ccw_ccp = umap.fit_transform(inits_orig[ccw_ccp_lo_eff_idxes])
# embs_hi_ccw_ccp = umap.fit_transform(inits_orig[ccw_ccp_hi_eff_idxes])

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=6)
colours = kmeans.fit_predict(embs_ccw_ccp)

plt.scatter(embs_ccw_ccp[:, 0], embs_ccw_ccp[:, 1], c=colours, s=3)
plt.colorbar()

In [ ]:
labels

Three clusters for effective:

- CC not human caused
- Not worried
- 

In [ ]:
from scipy.cluster.hierarchy import dendrogram, ward

In [ ]:
fig, ax = plt.subplots()
dendrogram(ward(inits_orig[ccw_ccp_lo_eff_idxes].T), labels=labels)
ax.tick_params(axis="x", rotation=90)

In [ ]:
eff = [0, 2, 5]
print(X[colours == 0].mean(axis=0))
print(X[colours == 2].mean(axis=0))
print(X[colours == 5].mean(axis=0))

In [ ]:
plt.scatter(embs_ccw_ccp[:, 0], embs_ccw_ccp[:, 1], c=Y, s=3)

In [ ]:
colours = inits_orig[ccw_ccp_hi_eff_idxes].mean(axis=1)
plt.scatter(embs_ccw_ccp[:, 0], embs_ccw_ccp[:, 1], c=colours, s=5, cmap="Spectral")
plt.gca().set_aspect("equal", "datalim")
plt.colorbar()

In [ ]:
labels

In [ ]:
colours = inits_orig[ccw_ccp_hi_eff_idxes][:, 7] > 0
plt.scatter(
    embs_ccw_ccp[:, 0],
    embs_ccw_ccp[:, 1],
    c=colours,
    s=5,
    cmap="Spectral",
    vmin=0,
    vmax=1,
)
plt.gca().set_aspect("equal", "datalim")
plt.colorbar()